# 03 - Patient-level splits (seed 42)

Splits are grouped so that no case is ever cut across train, val and test, with a fixed
seed (42). The grouping unit is the finest one each dataset supports: patient for
ThyroidXL and DDTI, nodule for Stanford AIMI (its release exposes no patient id, and every
frame of a nodule stays together), and image for TN3K, which is one image per case.

**Only the two open datasets ship with committed split CSVs** (`ddti_*.csv`, `tn3k_*.csv`).
The ThyroidXL and Stanford AIMI splits are not committed, because their filenames are
identifiers covered by those datasets' data-use agreements. You regenerate them here once
you have placed your approved copies; seed 42 makes that deterministic, so you get exactly
the splits the reported results use.


In [ ]:
# Always run from the repository root so every relative path resolves.
import os
from pathlib import Path
while not (Path.cwd() / 'pyproject.toml').exists() and Path.cwd() != Path.cwd().parent:
    os.chdir('..')
assert (Path.cwd() / 'pyproject.toml').exists(), 'run this notebook from inside the repo'
print('repo root:', Path.cwd())

### Regenerate
Reads `data/raw/extracted/...` and rewrites `data/splits/{dataset}_{split}.csv`.
Deterministic under seed 42 — identical to the committed CSVs.

In [ ]:
!python 00_setup/_lib/make_splits.py

### Verify against the committed splits

For the two open datasets the regenerated CSVs must match the committed ones bit for bit,
so `git status data/splits/` should report no changes to `ddti_*` or `tn3k_*`. The two
gated datasets will show up as new untracked files; that is expected, and `.gitignore`
keeps them from being committed.


In [ ]:
import subprocess
out = subprocess.run(['git', 'status', '--short', 'data/splits/'],
                     capture_output=True, text=True).stdout
open_changes = [l for l in out.splitlines() if 'ddti_' in l or 'tn3k_' in l]
if open_changes:
    print('the open-dataset splits differ from the committed ones:')
    print('\n'.join(open_changes))
else:
    print('open-dataset splits match the committed paper splits exactly')
gated = [l for l in out.splitlines() if 'thyroidxl_' in l or 'stanford' in l]
if gated:
    print(f'\n{len(gated)} gated split files regenerated locally (not committed, as expected)')